In [14]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))
from snake.snake_gym import Snake

from gymnasium.wrappers import FrameStackObservation, RecordVideo, RecordEpisodeStatistics
from gymnasium.vector import SyncVectorEnv

import numpy as np

import torch
import torch.nn as nn
from einops import rearrange, repeat
from einops.layers.torch import Rearrange
from torch.utils.tensorboard import SummaryWriter

import time
import matplotlib.pyplot as plt

In [15]:
model_path = "../models/ppo_vit"
prev_version = "v1"
model_version = "v1"

train_video_dir = f"../data/ppo_vit/v1"
train_episodes = 500
num_steps = 2048
num_envs = 8
stack_size = 4

In [16]:
def make_env(idx: int, capture_video: bool = False):
    def thunk():
        env = Snake(truncation_steps=25000)
        
        env = FrameStackObservation(env, stack_size=stack_size)        
        env = RecordEpisodeStatistics(env=env)
        
        if capture_video and idx == 0:
            env = RecordVideo(env, train_video_dir, episode_trigger=lambda episode_id: episode_id % 100 == 0)
        return env
    return thunk

In [17]:
envs = SyncVectorEnv([make_env(idx, capture_video=False) for idx in range(num_envs)])

In [18]:
action_size = envs.single_action_space.n
action_size

np.int64(4)

In [19]:
envs.single_observation_space

Box(0.0, 3.0, (4, 30, 30), float32)

In [20]:
obs, _ = envs.reset()
obs.shape

(8, 4, 30, 30)

In [21]:
device = "cuda" if torch.cuda.is_available() else "cpu"
device

'cuda'

In [22]:
def pair(t):
    return t if isinstance(t, tuple) else (t, t)

class FeedForward(nn.Module):
    def __init__(self, dim, hidden_dim, dropout = 0.):
        super().__init__()
        self.net = nn.Sequential(
            nn.LayerNorm(dim),
            nn.Linear(dim, hidden_dim),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, dim),
            nn.Dropout(dropout)
        )

    def forward(self, x):
        return self.net(x)


class Attention(nn.Module):
    def __init__(self, dim, heads = 8, dim_head = 64, dropout = 0.):
        super().__init__()
        inner_dim = dim_head *  heads
        project_out = not (heads == 1 and dim_head == dim)

        self.heads = heads
        self.scale = dim_head ** -0.5

        self.norm = nn.LayerNorm(dim)

        self.attend = nn.Softmax(dim = -1)
        self.dropout = nn.Dropout(dropout)

        self.to_qkv = nn.Linear(dim, inner_dim * 3, bias = False)

        self.to_out = nn.Sequential(
            nn.Linear(inner_dim, dim),
            nn.Dropout(dropout)
        ) if project_out else nn.Identity()
        
    def forward(self, x):
        x = self.norm(x)

        qkv = self.to_qkv(x).chunk(3, dim = -1)
        q, k, v = map(lambda t: rearrange(t, 'b n (h d) -> b h n d', h = self.heads), qkv)

        dots = torch.matmul(q, k.transpose(-1, -2)) * self.scale

        attn = self.attend(dots)
        attn = self.dropout(attn)

        out = torch.matmul(attn, v)
        out = rearrange(out, 'b h n d -> b n (h d)')
        return self.to_out(out)


class Transformer(nn.Module):
    def __init__(self, dim, depth, heads, dim_head, mlp_dim, dropout = 0.):
        super().__init__()
        self.norm = nn.LayerNorm(dim)
        self.layers = nn.ModuleList([])

        for _ in range(depth):
            self.layers.append(nn.ModuleList([
                Attention(dim, heads = heads, dim_head = dim_head, dropout = dropout),
                FeedForward(dim, mlp_dim, dropout = dropout)
            ]))

    def forward(self, x):
        for attn, ff in self.layers:
            x = attn(x) + x
            x = ff(x) + x

        return self.norm(x)

class ViT(nn.Module):
    def __init__(self, *, image_size, patch_size, dim, depth, heads, mlp_dim, pool = 'cls', channels = 3, dim_head = 64, dropout = 0., emb_dropout = 0.):
        super().__init__()
        image_height, image_width = pair(image_size)
        self.patch_size = patch_height, patch_width = pair(patch_size)

        assert image_height % patch_height == 0 and image_width % patch_width == 0, 'Image dimensions must be divisible by the patch size.'

        num_patches = (image_height // patch_height) * (image_width // patch_width)
        patch_dim = channels * patch_height * patch_width

        assert pool in {'cls', 'mean'}, 'pool type must be either cls (cls token) or mean (mean pooling)'
        num_cls_tokens = 1 if pool == 'cls' else 0

        self.to_patch_embedding = nn.Sequential(
            Rearrange('b c (h p1) (w p2) -> b (h w) (p1 p2 c)', p1 = patch_height, p2 = patch_width),
            nn.LayerNorm(patch_dim),
            nn.Linear(patch_dim, dim),
            nn.LayerNorm(dim),
        )

        self.cls_token = nn.Parameter(torch.randn(num_cls_tokens, dim))
        self.pos_embedding = nn.Parameter(torch.randn(num_patches + num_cls_tokens, dim))

        self.dropout = nn.Dropout(emb_dropout)

        self.transformer = Transformer(dim, depth, heads, dim_head, mlp_dim, dropout)

        self.pool = pool

    def forward(self, img):
        batch = img.shape[0]
        x = self.to_patch_embedding(img)

        cls_tokens = repeat(self.cls_token, '... d -> b ... d', b = batch)
        x = torch.cat((cls_tokens, x), dim = 1)

        seq = x.shape[1]

        x = x + self.pos_embedding[:seq]
        x = self.dropout(x)

        x = self.transformer(x)
        
        x = x[:, 0]

        return x

def layer_init(layer, std=np.sqrt(2), bias_const=0.0):
    torch.nn.init.orthogonal_(layer.weight, std)
    torch.nn.init.constant_(layer.bias, bias_const)
    return layer

class Agent(nn.Module):
    def __init__(self, n_hid, n_out):
        super().__init__()
        
        self.vit = ViT(
            image_size=(30, 30),
            patch_size=(3, 3),
            dim=n_hid,
            dim_head=n_hid // 4,
            depth=4,
            heads=4,
            mlp_dim=256,
            channels=stack_size
        )
        
        self.actor = layer_init(nn.Linear(n_hid, n_out), std=0.01)
        self.critic = layer_init(nn.Linear(n_hid, 1), std=1)                

    def get_value(self, x):
        return self.critic(self.vit(x))
    
    def get_action(self, x):
        hidden = self.vit(x)
        logits = self.actor(hidden)
        
        action = torch.argmax(logits, dim=1)
        
        return action.item()

    def get_action_and_value(self, x, action=None):
        hidden = self.vit(x)
        logits = self.actor(hidden)
        probs = torch.distributions.Categorical(logits=logits)
        if action is None:
            action = probs.sample()
        return action, probs.log_prob(action), probs.entropy(), self.critic(hidden)

In [23]:
x = torch.randn(8, 4, 30, 30)

sample_agent = Agent(n_hid=128, n_out=action_size)

action, log_prob, entropy, value = sample_agent.get_action_and_value(x)
print(action.shape)
print(log_prob.shape)
print(entropy.shape)
print(value.shape)

torch.Size([8])
torch.Size([8])
torch.Size([8])
torch.Size([8, 1])


In [24]:
def train(
    envs,
    model,
    optimizer,
    updates,
    update_epochs,
    num_steps,
    target_kl = 0.03,
    mini_batch_size = 64,
    gae_lambda = 0.95,
    clip_coef = 0.2,
    ent_coef = 0.01,
    vf_coef = 0.5,
    max_grad_norm = 0.5,
    norm_advantages = False,
    clip_vloss = False,
    lr_decay = False,
    gamma = 0.999,
    initial_lr = 1e-3,
    min_lr = 1e-5
):
    
    total_timesteps = updates * num_steps * num_envs
    
    writer = SummaryWriter(f"runs_vit/snake-{model_version}")
    
    batch_size = num_steps * num_envs
    
    obs = torch.zeros((num_steps, num_envs) + envs.single_observation_space.shape).to(device)
    actions = torch.zeros((num_steps, num_envs) + envs.single_action_space.shape, dtype=torch.long).to(device)
    logprobs = torch.zeros((num_steps, num_envs)).to(device)
    rewards = torch.zeros((num_steps, num_envs)).to(device)
    dones = torch.zeros((num_steps, num_envs)).to(device)
    values = torch.zeros((num_steps, num_envs)).to(device)
    
    global_step = 0
    next_obs, _ = envs.reset()
    next_obs = torch.Tensor(next_obs).to(device)
    next_done = torch.zeros(num_envs).to(device)
    start_time = time.time()
    
    for episode in range(1, updates + 1):      
        total_reward = 0
        
        if lr_decay:
            frac = max(
                0.0,
                1.0 - global_step / total_timesteps
            )

            lrnow = min_lr + frac * (initial_lr - min_lr)

            optimizer.param_groups[0]["lr"] = lrnow
        
        for step in range(0, num_steps):
            global_step += num_envs
            obs[step] = next_obs
            dones[step] = next_done
            
            with torch.no_grad():
                action, logprob, _, value = model.get_action_and_value(next_obs)
                values[step] = value.flatten()
            actions[step] = action
            logprobs[step] = logprob
            
            next_obs, reward, terminations, truncations, infos = envs.step(action.cpu().numpy())
            next_done = np.logical_or(terminations, truncations)
            rewards[step] = torch.tensor(reward).to(device).view(-1)
            total_reward += float(reward.mean())
            next_obs, next_done = torch.Tensor(next_obs).to(device), torch.tensor(next_done, dtype=torch.float32).to(device)
            
            if "episode" in infos:
                ep = infos["episode"]

                for reward, finished, length in zip(
                    ep["r"],
                    ep["_r"],
                    ep["l"],
                ):
                    if finished:
                        writer.add_scalar("charts/episodic_return", reward, global_step)
                        writer.add_scalar("charts/episodic_length", length, global_step)
                
                                    
        with torch.no_grad():
            next_value = model.get_value(next_obs).reshape(1, -1)
            advantages = torch.zeros_like(rewards).to(device)
            lastgaelam = 0
            for t in reversed(range(num_steps)):
                if t == num_steps - 1:
                    next_nonterminal = 1.0 - next_done
                    next_values = next_value
                else:
                    next_nonterminal = 1.0 - dones[t + 1]
                    next_values = values[t + 1]
                delta = rewards[t] + gamma * next_values * next_nonterminal - values[t]
                advantages[t] = lastgaelam = delta + gamma * gae_lambda * next_nonterminal * lastgaelam
            returns = advantages + values
         
        b_obs = obs.reshape((-1,) + envs.single_observation_space.shape)
        b_logprobs = logprobs.reshape(-1)
        b_actions = actions.reshape((-1,) + envs.single_action_space.shape)
        b_advanatages = advantages.reshape(-1)
        b_returns = returns.reshape(-1)
        b_values = values.reshape(-1)
        
        b_inds = np.arange(batch_size)
        clipfracs = []
        for epoch in range(update_epochs):
            np.random.shuffle(b_inds)
            for start in range(0, batch_size, mini_batch_size):
                end = start + mini_batch_size
                mb_inds = b_inds[start:end]
                
                _, newlogprob, entropy, newvalue = model.get_action_and_value(b_obs[mb_inds], b_actions.long()[mb_inds])
                logratio = newlogprob - b_logprobs[mb_inds]
                ratio = logratio.exp()
                
                with torch.no_grad():
                    old_approx_kl = (-logratio).mean()
                    approx_kl = ((ratio - 1) - logratio).mean()
                    clipfracs += [((ratio - 1.0).abs() > clip_coef).float().mean().item()]
                    
                mb_advantages = b_advanatages[mb_inds]
                if norm_advantages: 
                    mb_advantages = (mb_advantages-mb_advantages.mean()) / (mb_advantages.std() + 1e-8)
                
                pg_loss1 = -mb_advantages * ratio
                pg_loss2 = -mb_advantages * torch.clamp(ratio, 1 - clip_coef, 1 + clip_coef)
                pg_loss = torch.max(pg_loss1, pg_loss2).mean()
                
                new_value = newvalue.view(-1)

                if clip_vloss:
                    v_loss_unclipped = (
                        new_value - b_returns[mb_inds]
                    ) ** 2

                    v_clipped = (
                        b_values[mb_inds]
                        + torch.clamp(
                            new_value - b_values[mb_inds],
                            -clip_coef,
                            clip_coef,
                        )
                    )

                    v_loss_clipped = (
                        v_clipped - b_returns[mb_inds]
                    ) ** 2

                    v_loss_max = torch.max(
                        v_loss_unclipped,
                        v_loss_clipped
                    )

                    v_loss = 0.5 * v_loss_max.mean()

                else:
                    v_loss = 0.5 * (
                        (new_value - b_returns[mb_inds]) ** 2
                    ).mean()
                    
                entropy_loss = entropy.mean()
                loss = pg_loss - ent_coef * entropy_loss + v_loss * vf_coef
                
                optimizer.zero_grad()
                loss.backward()
                nn.utils.clip_grad_norm_(model.parameters(), max_grad_norm)
                optimizer.step()
                
            if target_kl is not None and approx_kl > target_kl:
                break
            
        y_pred, y_true = b_values.cpu().numpy(), b_returns.cpu().numpy()
        var_y = np.var(y_true)
        explained_var = np.nan if var_y == 0 else 1 - np.var(y_true - y_pred) / var_y
        
        writer.add_scalar("charts/learning_rate", optimizer.param_groups[0]["lr"], global_step)
        writer.add_scalar("losses/value_loss", v_loss.item(), global_step)
        writer.add_scalar("losses/policy_loss", pg_loss.item(), global_step)
        writer.add_scalar("losses/entropy", entropy_loss.item(), global_step)
        writer.add_scalar("losses/old_approx_kl", old_approx_kl.item(), global_step)
        writer.add_scalar("losses/approx_kl", approx_kl.item(), global_step)
        writer.add_scalar("losses/clipfrac", np.mean(clipfracs), global_step)
        writer.add_scalar("losses/explained_variance", explained_var, global_step)
        writer.add_scalar("charts/mean_rollout_reward", total_reward, global_step) 
        print(f"\rSPS: {int(global_step / (time.time() - start_time))} | episode: {episode} | mean rollout reward: {total_reward} | global_step: {global_step}")
        writer.add_scalar("charts/SPS", int(global_step / (time.time() - start_time)), global_step)
            
    envs.close()
    writer.close()

In [25]:
lr = 3e-4
agent = Agent(128, action_size).to(device)
# agent.load_state_dict(torch.load(f"{model_path}/agent-{prev_version}.pth"))

optimizer = torch.optim.AdamW(agent.parameters(), lr=lr, eps=1e-5, weight_decay=1e-4)
# optimizer.load_state_dict(torch.load(f"{model_path}/optimizer-{prev_version}.pth"))

# for pg in optimizer.param_groups:
#     pg["lr"] = lr
#     print(f"current lr: {pg["lr"]}")

In [26]:
train(
    envs=envs,
    model=agent,
    optimizer=optimizer,
    updates=train_episodes,
    
    update_epochs=4,
    num_steps=num_steps,
    target_kl=0.03,
    mini_batch_size=512,
    gae_lambda=0.95,
    clip_coef=0.2,
    ent_coef=0.01,
    vf_coef=0.5,
    max_grad_norm=0.5,
    
    norm_advantages=True,
    clip_vloss=True,
    lr_decay=False,
    
    gamma=0.999,
    initial_lr=lr,
    min_lr=1e-4
)

SPS: 690 | episode: 1 | mean rollout reward: 12.125 | global_step: 16384
SPS: 657 | episode: 2 | mean rollout reward: 16.0 | global_step: 32768
SPS: 634 | episode: 3 | mean rollout reward: 26.125 | global_step: 49152
SPS: 608 | episode: 4 | mean rollout reward: 34.25 | global_step: 65536
SPS: 543 | episode: 5 | mean rollout reward: 49.875 | global_step: 81920
SPS: 513 | episode: 6 | mean rollout reward: 58.0 | global_step: 98304
SPS: 509 | episode: 7 | mean rollout reward: 54.625 | global_step: 114688
SPS: 503 | episode: 8 | mean rollout reward: 55.375 | global_step: 131072
SPS: 489 | episode: 9 | mean rollout reward: 51.125 | global_step: 147456
SPS: 479 | episode: 10 | mean rollout reward: 51.0 | global_step: 163840
SPS: 477 | episode: 11 | mean rollout reward: 48.625 | global_step: 180224
SPS: 466 | episode: 12 | mean rollout reward: 44.25 | global_step: 196608
SPS: 465 | episode: 13 | mean rollout reward: 43.875 | global_step: 212992
SPS: 466 | episode: 14 | mean rollout reward: 42

KeyboardInterrupt: 

In [ ]:
torch.save(agent.state_dict(), f"{model_path}/agent-{model_version}.pth")
torch.save(optimizer.state_dict(), f"{model_path}/optimizer-{model_version}.pth")